# Notebook 3: Deep Ensemble Comparison and Advanced Calibration Analysis

This notebook extends the uncertainty quantification pipeline from Notebooks 1--2 by comparing
two complementary approaches to predictive uncertainty in deep MRI reconstruction:
**MC Dropout** and **Deep Ensembles**. While MC Dropout approximates Bayesian inference
by sampling from an implicit variational posterior at test time, Deep Ensembles capture
epistemic uncertainty through disagreement among independently trained models.

We evaluate uncertainty quality using three complementary calibration metrics (ECE, AUSE,
Pearson r), ablate the contribution of data consistency (DC) layers, and investigate
whether reconstruction uncertainty is predictive of downstream segmentation failures --
a clinically important property for deployment in diagnostic pipelines.

**Key questions addressed:**
1. Which uncertainty method produces better-calibrated uncertainty maps?
2. How much does data consistency contribute to reconstruction fidelity?
3. Can reconstruction uncertainty serve as a proxy for downstream task failure?

## 1. Environment Setup and Model Loading

We load the main reconstruction model (trained with MC Dropout enabled),
three independently trained ensemble members, and a segmentation model
used to assess downstream task impact.

In [16]:
# Importsimport os, sys, json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats

# Add src to path
sys.path.insert(0, '/root/IX-Medical-Imaging/src')
from data import get_dataloaders, MRIReconDataset
from models import ReconUNet, SegmentationUNet
from losses import compute_psnr, compute_ssim, compute_nmse

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '/root/IX-Medical-Imaging/data/processed_data'
CKPT_DIR = '/root/IX-Medical-Imaging/checkpoints'
FIG_DIR = '/root/IX-Medical-Imaging/latex/figures'
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Device: {DEVICE}")

Device: cuda


In [17]:
# Load test data# We use 4x accelerated MRI data, matching the training configuration.
print("Loading test data...")
_, _, test_loader = get_dataloaders(DATA_ROOT, modality='mr', acceleration=4,
                                     batch_size=1, num_workers=2)
print(f"Test set: {len(test_loader)} samples")

Loading test data...
Test set: 236 samples


In [18]:
# Model loading utilities# MODEL_PARAMS match the Optuna-optimised hyperparameters from training:
# dropout_rate=0.112 enables meaningful MC Dropout variance,
# use_dc=True enables data consistency cascades for k-space enforcement.
MODEL_PARAMS = dict(in_channels=1, out_channels=1, base_features=32,
                    dropout_rate=0.112, use_dc=True, num_dc_cascades=3)

def load_recon_model(path):
    """Load ReconUNet from checkpoint dict."""
    ckpt = torch.load(path, map_location=DEVICE)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    m = ReconUNet(**MODEL_PARAMS).to(DEVICE)
    m.load_state_dict(state)
    return m

def load_seg_model(path):
    """Load SegmentationUNet from checkpoint dict."""
    ckpt = torch.load(path, map_location=DEVICE)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    m = SegmentationUNet(in_channels=1, num_classes=8, base_features=32).to(DEVICE)
    m.load_state_dict(state)
    return m

In [19]:
# Load all modelsprint("Loading main model...")
main_model = load_recon_model(f'{CKPT_DIR}/final_R4/best_model_R4.pth')
main_model.eval()

print("Loading ensemble models...")
ensemble_models = []
for i in range(3):
    m = load_recon_model(f'{CKPT_DIR}/ensemble_{i}_R4/best_model_R4.pth')
    m.eval()
    ensemble_models.append(m)
print(f"Loaded {len(ensemble_models)} ensemble members")

print("Loading segmentation model...")
seg_model = load_seg_model(f'{CKPT_DIR}/seg_model.pth')
seg_model.eval()

Loading main model...
Loading ensemble models...
Loaded 3 ensemble members
Loading segmentation model...


SegmentationUNet(
  (enc1): ConvBlock(
    (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (dropout): Identity()
    (relu): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (enc2): ConvBlock(
    (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (dropout): Identity()
    (relu): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (enc3): ConvBlock(


## 2. Inference: MC Dropout vs Deep Ensemble

**MC Dropout** (Gal & Ghahramani, 2016) approximates Bayesian inference by keeping
dropout active at test time and running T=30 stochastic forward passes. The variance
across passes captures epistemic (model) uncertainty. This is computationally cheap
since it requires only one trained model.

**Deep Ensembles** (Lakshminarayanan et al., 2017) train M=3 models independently
from different random initialisations. Disagreement across ensemble members captures
both epistemic uncertainty and sensitivity to initialisation. Ensembles are widely
considered a strong baseline for uncertainty estimation, though they incur M-fold
training cost.

Comparing both methods on identical test data reveals whether the cheaper MC Dropout
approach yields comparable uncertainty quality to the more expensive ensemble strategy.

In [20]:
# Run inference on the full test set# MC Dropout with T=30 forward passes, and Deep Ensemble with M=3 members.
print("\n=== Running MC Dropout inference (T=30) ===")
MC_T = 30
all_mc_means, all_mc_stds = [], []
all_ens_means, all_ens_stds = [], []
all_targets, all_undersampled, all_errors_mc, all_errors_ens = [], [], [], []
all_labels = []
all_psnr_mc, all_psnr_ens, all_ssim_mc, all_ssim_ens = [], [], [], []

for i, batch in enumerate(test_loader):
    us = batch['undersampled'].to(DEVICE)
    ks = batch['kspace'].to(DEVICE)
    mk = batch['mask'].to(DEVICE)
    tgt = batch['target'].to(DEVICE)
    lbl = batch['label']

    # MC Dropout: T stochastic forward passes with dropout enabled
    mc_mean, mc_std = main_model.mc_predict(us, ks, mk, num_samples=MC_T)

    # Deep Ensemble: deterministic forward pass through each member
    with torch.no_grad():
        ens_preds = torch.stack([m(us, ks, mk) for m in ensemble_models], dim=0)
    ens_mean = ens_preds.mean(0)
    ens_std = ens_preds.std(0)

    # Store results
    all_mc_means.append(mc_mean.cpu())
    all_mc_stds.append(mc_std.cpu())
    all_ens_means.append(ens_mean.cpu())
    all_ens_stds.append(ens_std.cpu())
    all_targets.append(tgt.cpu())
    all_undersampled.append(us.cpu())
    all_labels.append(lbl)

    # Pixel-wise absolute errors for calibration analysis
    err_mc = torch.abs(mc_mean - tgt)
    err_ens = torch.abs(ens_mean - tgt)
    all_errors_mc.append(err_mc.cpu())
    all_errors_ens.append(err_ens.cpu())

    # Per-sample image quality metrics
    all_psnr_mc.append(compute_psnr(mc_mean, tgt).item())
    all_psnr_ens.append(compute_psnr(ens_mean, tgt).item())
    all_ssim_mc.append(compute_ssim(mc_mean, tgt).item())
    all_ssim_ens.append(compute_ssim(ens_mean, tgt).item())

    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(test_loader)} slices")

print(f"MC Dropout: PSNR={np.mean(all_psnr_mc):.2f} dB, SSIM={np.mean(all_ssim_mc):.4f}")
print(f"Ensemble:   PSNR={np.mean(all_psnr_ens):.2f} dB, SSIM={np.mean(all_ssim_ens):.4f}")


=== Running MC Dropout inference (T=30) ===
  Processed 50/236 slices
  Processed 100/236 slices
  Processed 150/236 slices
  Processed 200/236 slices
MC Dropout: PSNR=31.95 dB, SSIM=0.8893
Ensemble:   PSNR=32.09 dB, SSIM=0.8913


In [21]:
# Concatenate results into arrays for analysismc_stds_all = torch.cat(all_mc_stds, dim=0).squeeze().numpy()    # (N, H, W)
ens_stds_all = torch.cat(all_ens_stds, dim=0).squeeze().numpy()
errors_mc_all = torch.cat(all_errors_mc, dim=0).squeeze().numpy()
errors_ens_all = torch.cat(all_errors_ens, dim=0).squeeze().numpy()
targets_all = torch.cat(all_targets, dim=0).squeeze().numpy()
mc_means_all = torch.cat(all_mc_means, dim=0).squeeze().numpy()
ens_means_all = torch.cat(all_ens_means, dim=0).squeeze().numpy()

## 3. Calibration Metrics (ECE, AUSE, Pearson r)

A good uncertainty estimate should be **calibrated**: regions where the model
predicts high uncertainty should indeed exhibit high error, and vice versa.
We assess calibration using three complementary metrics:

- **Expected Calibration Error (ECE):** Measures the average discrepancy between
  predicted uncertainty and observed error across bins. An ECE of 0 indicates
  perfect calibration. This is the regression analogue of the classification ECE
  introduced by Naeini et al. (2015).

- **Area Under the Sparsification Error (AUSE):** Evaluates whether removing
  high-uncertainty pixels preferentially removes high-error pixels. The gap
  between the uncertainty-guided sparsification curve and the oracle (error-guided)
  curve quantifies how well uncertainty ranks errors. Lower AUSE is better.
  Introduced for image regression by Ilg et al. (2018).

- **Pearson correlation coefficient (r):** Measures linear correlation between
  pixel-wise uncertainty and pixel-wise absolute error. Higher r indicates that
  uncertainty magnitude tracks error magnitude, a necessary (but not sufficient)
  condition for calibration.

Together, these metrics test different aspects of uncertainty quality: calibration
(ECE), ranking (AUSE), and correlation (Pearson r).

In [22]:
# Calibration metric implementationsprint("\n=== Computing Calibration Metrics ===")

def compute_ece(uncertainty, error, n_bins=15):
    """Expected Calibration Error for regression.
    Bins pixels by normalised uncertainty, then measures the weighted average
    absolute difference between mean uncertainty and mean error per bin.
    """
    unc_flat = uncertainty.flatten()
    err_flat = error.flatten()
    # Normalize both to [0, 1] for comparable scales
    unc_norm = unc_flat / (unc_flat.max() + 1e-10)
    err_norm = err_flat / (err_flat.max() + 1e-10)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total = len(unc_norm)
    for i in range(n_bins):
        mask = (unc_norm >= bins[i]) & (unc_norm < bins[i+1])
        if mask.sum() == 0:
            continue
        avg_unc = unc_norm[mask].mean()
        avg_err = err_norm[mask].mean()
        ece += (mask.sum() / total) * abs(avg_unc - avg_err)
    return ece

def compute_ause(uncertainty, error, n_fractions=50):
    """Area Under Sparsification Error curve.
    Progressively removes pixels in order of decreasing uncertainty (or error for oracle),
    tracking how residual MSE decreases. The gap between uncertainty-guided and oracle
    curves measures how well uncertainty ranks errors.
    """
    unc_flat = uncertainty.flatten()
    err_flat = (error.flatten()) ** 2  # Use MSE for sparsification

    # Sort indices: highest uncertainty/error first
    unc_sorted_idx = np.argsort(-unc_flat)
    oracle_sorted_idx = np.argsort(-err_flat)

    fractions = np.linspace(0, 0.95, n_fractions)
    unc_mse_curve, oracle_mse_curve = [], []

    for f in fractions:
        n_remove = int(f * len(unc_flat))
        if n_remove == 0:
            unc_mse_curve.append(err_flat.mean())
            oracle_mse_curve.append(err_flat.mean())
        else:
            remaining_unc = np.delete(err_flat, unc_sorted_idx[:n_remove])
            remaining_oracle = np.delete(err_flat, oracle_sorted_idx[:n_remove])
            unc_mse_curve.append(remaining_unc.mean() if len(remaining_unc) > 0 else 0)
            oracle_mse_curve.append(remaining_oracle.mean() if len(remaining_oracle) > 0 else 0)

    ause = np.trapezoid(unc_mse_curve, fractions) - np.trapezoid(oracle_mse_curve, fractions)
    return ause, fractions, unc_mse_curve, oracle_mse_curve


=== Computing Calibration Metrics ===


In [23]:
# Compute calibration metrics for both methods# MC Dropout
mc_ece = compute_ece(mc_stds_all, errors_mc_all)
mc_ause, mc_fracs, mc_spars, mc_oracle = compute_ause(mc_stds_all, errors_mc_all)
mc_pearson = np.corrcoef(mc_stds_all.flatten(), errors_mc_all.flatten())[0, 1]

# Deep Ensemble
ens_ece = compute_ece(ens_stds_all, errors_ens_all)
ens_ause, ens_fracs, ens_spars, ens_oracle = compute_ause(ens_stds_all, errors_ens_all)
ens_pearson = np.corrcoef(ens_stds_all.flatten(), errors_ens_all.flatten())[0, 1]

# Random baseline for sparsification (removing pixels at random should not reduce MSE)
_, rand_fracs, _, _ = compute_ause(np.random.rand(*mc_stds_all.shape), errors_mc_all)
random_spars = []
err_flat_all = (errors_mc_all.flatten()) ** 2
for f in np.linspace(0, 0.95, 50):
    n_keep = int((1 - f) * len(err_flat_all))
    if n_keep > 0:
        random_spars.append(err_flat_all.mean())
    else:
        random_spars.append(0)

print(f"\nMC Dropout:  ECE={mc_ece:.4f}, AUSE={mc_ause:.6f}, Pearson r={mc_pearson:.4f}")
print(f"Ensemble:    ECE={ens_ece:.4f}, AUSE={ens_ause:.6f}, Pearson r={ens_pearson:.4f}")


MC Dropout:  ECE=0.0342, AUSE=0.000093, Pearson r=0.5901
Ensemble:    ECE=0.0170, AUSE=0.000129, Pearson r=0.4934


## 4. Data Consistency Ablation

Data consistency (DC) layers enforce agreement between the reconstructed image
and the acquired k-space measurements at observed frequency locations. This is
a physics-informed constraint that prevents the network from hallucinating
features inconsistent with the measured data.

This ablation quantifies the DC contribution by comparing reconstruction quality
with and without DC at inference time (using the same trained model). A large
quality drop without DC would confirm that the physics prior is critical and
not merely a regularisation convenience. This is important for trustworthiness:
if DC meaningfully improves fidelity, then uncertainty estimates from the
DC-enabled model are more likely to be meaningful because the reconstructions
themselves are more faithful to the acquired data.

In [24]:
# DC Ablation: compare with-DC vs without-DC inferenceprint("\n=== DC Ablation (disabling DC at inference) ===")
main_model.eval()
dc_psnrs, nodc_psnrs = [], []
dc_ssims, nodc_ssims = [], []

for i, batch in enumerate(test_loader):
    us = batch['undersampled'].to(DEVICE)
    ks = batch['kspace'].to(DEVICE)
    mk = batch['mask'].to(DEVICE)
    tgt = batch['target'].to(DEVICE)

    with torch.no_grad():
        # With DC: full model with k-space consistency enforcement
        recon_dc = main_model(us, ks, mk)
        dc_psnrs.append(compute_psnr(recon_dc, tgt).item())
        dc_ssims.append(compute_ssim(recon_dc, tgt).item())

        # Without DC: pass None for kspace/mask to bypass DC layers
        recon_nodc = main_model(us, None, None)
        nodc_psnrs.append(compute_psnr(recon_nodc, tgt).item())
        nodc_ssims.append(compute_ssim(recon_nodc, tgt).item())

print(f"With DC:    PSNR={np.mean(dc_psnrs):.2f} dB, SSIM={np.mean(dc_ssims):.4f}")
print(f"Without DC: PSNR={np.mean(nodc_psnrs):.2f} dB, SSIM={np.mean(nodc_ssims):.4f}")
print(f"DC benefit: +{np.mean(dc_psnrs)-np.mean(nodc_psnrs):.2f} dB PSNR")


=== DC Ablation (disabling DC at inference) ===
With DC:    PSNR=31.90 dB, SSIM=0.8886
Without DC: PSNR=23.84 dB, SSIM=0.6192
DC benefit: +8.06 dB PSNR


## 5. Uncertainty-Segmentation Failure Correlation

In clinical MRI pipelines, reconstruction is rarely the final step. Downstream
tasks such as tissue segmentation depend on reconstruction quality. If
reconstruction uncertainty can predict where segmentation will fail, clinicians
gain an actionable signal: high-uncertainty regions warrant manual review.

We measure this by:
1. Running the segmentation model on both the ground truth and the reconstruction.
2. Computing a binary segmentation error map (where reconstruction-based
   segmentation disagrees with ground-truth labels).
3. Computing the Pearson correlation between the reconstruction uncertainty map
   and the segmentation error map.

A positive correlation indicates that reconstruction uncertainty is informative
about downstream task failure -- a key property for trustworthy deployment.

In [25]:
# Compute uncertainty-segmentation correlationprint("\n=== Uncertainty-Segmentation Correlation ===")
seg_correlations_mc = []
seg_correlations_ens = []
seg_error_maps = []

for i, batch in enumerate(test_loader):
    tgt = batch['target'].to(DEVICE)
    lbl = batch['label'].to(DEVICE)  # Ground-truth segmentation labels (B, H, W)

    # Reconstruct from stored results
    mc_recon = torch.from_numpy(mc_means_all[i:i+1]).unsqueeze(1).to(DEVICE)
    ens_recon = torch.from_numpy(ens_means_all[i:i+1]).unsqueeze(1).to(DEVICE)

    with torch.no_grad():
        # Segmentation on ground truth vs on reconstructed images
        seg_on_gt = seg_model(tgt).argmax(1)       # Reference segmentation
        seg_on_mc = seg_model(mc_recon.float()).argmax(1)
        seg_on_ens = seg_model(ens_recon.float()).argmax(1)

    # Binary error maps: where segmentation on reconstruction differs from GT labels
    seg_err_mc = (seg_on_mc != lbl).float().cpu().numpy().squeeze()
    seg_err_ens = (seg_on_ens != lbl).float().cpu().numpy().squeeze()

    mc_unc = mc_stds_all[i]
    ens_unc = ens_stds_all[i]

    # Pearson correlation (only computed where there is variation in both signals)
    if seg_err_mc.std() > 0 and mc_unc.std() > 0:
        r_mc = np.corrcoef(mc_unc.flatten(), seg_err_mc.flatten())[0, 1]
        seg_correlations_mc.append(r_mc)

    if seg_err_ens.std() > 0 and ens_unc.std() > 0:
        r_ens = np.corrcoef(ens_unc.flatten(), seg_err_ens.flatten())[0, 1]
        seg_correlations_ens.append(r_ens)

    seg_error_maps.append(seg_err_mc)

mean_seg_corr_mc = np.mean(seg_correlations_mc)
mean_seg_corr_ens = np.mean(seg_correlations_ens)
print(f"MC Dropout uncertainty-seg error Pearson r: {mean_seg_corr_mc:.4f}")
print(f"Ensemble uncertainty-seg error Pearson r:   {mean_seg_corr_ens:.4f}")


=== Uncertainty-Segmentation Correlation ===
MC Dropout uncertainty-seg error Pearson r: 0.1926
Ensemble uncertainty-seg error Pearson r:   0.1564


## 6. Publication Figures

We generate three figures for the LNCS paper:

- **Figure 13:** Side-by-side comparison of MC Dropout and Deep Ensemble outputs,
  including reconstruction means, uncertainty maps, error maps, sparsification
  curves, and uncertainty-error scatter plots. This figure directly supports the
  claim that both methods produce meaningful uncertainty while highlighting
  differences in spatial uncertainty patterns.

- **Figure 14:** Reliability diagrams (calibration plots) for both methods.
  These bin predictions by uncertainty level and compare predicted vs observed
  error, visually assessing calibration quality beyond the scalar ECE value.

- **Figure 15:** Uncertainty-segmentation failure correlation visualisation.
  Shows that high reconstruction uncertainty spatially coincides with
  segmentation errors, supporting clinical relevance of uncertainty maps.

All figures use serif fonts at 9pt to match the LNCS template typography.

In [26]:
# Select a representative sample (median PSNR quality)psnr_array = np.array(all_psnr_mc)
sample_idx = np.argsort(psnr_array)[len(psnr_array) // 2]

plt.rcParams.update({
    'font.size': 9, 'font.family': 'serif',
    'axes.labelsize': 9, 'axes.titlesize': 10,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'figure.dpi': 300
})

In [27]:
# Figure 13: Ensemble vs MC Dropout Comparisonfig = plt.figure(figsize=(10, 7))
gs = gridspec.GridSpec(3, 4, hspace=0.35, wspace=0.15,
                       height_ratios=[1, 1, 1.2])

s = sample_idx
gt = targets_all[s]
mc_rec = mc_means_all[s]
mc_unc = mc_stds_all[s]
mc_err = errors_mc_all[s]
ens_rec = ens_means_all[s]
ens_unc = ens_stds_all[s]
ens_err = errors_ens_all[s]

# Row 1: MC Dropout results
titles_r1 = ['Ground Truth', 'MC Dropout Mean', 'MC Dropout Uncertainty', 'Absolute Error']
imgs_r1 = [gt, mc_rec, mc_unc, mc_err]
cmaps_r1 = ['gray', 'gray', 'magma', 'hot']
for j, (img, title, cmap) in enumerate(zip(imgs_r1, titles_r1, cmaps_r1)):
    ax = fig.add_subplot(gs[0, j])
    im = ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    if j >= 2:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 2: Deep Ensemble results
titles_r2 = ['Ground Truth', 'Ensemble Mean', 'Ensemble Uncertainty', 'Absolute Error']
imgs_r2 = [gt, ens_rec, ens_unc, ens_err]
cmaps_r2 = ['gray', 'gray', 'magma', 'hot']
for j, (img, title, cmap) in enumerate(zip(imgs_r2, titles_r2, cmaps_r2)):
    ax = fig.add_subplot(gs[1, j])
    im = ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    if j >= 2:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 3: Sparsification curves and uncertainty-error scatter
ax_spars = fig.add_subplot(gs[2, :2])
ax_spars.plot(mc_fracs, mc_spars, 'b-', linewidth=1.5, label='MC Dropout')
ax_spars.plot(ens_fracs, ens_spars, 'r-', linewidth=1.5, label='Deep Ensemble')
ax_spars.plot(mc_fracs, mc_oracle, 'k--', linewidth=1, label='Oracle', alpha=0.7)
ax_spars.plot(mc_fracs, random_spars, 'gray', linewidth=1, linestyle=':', label='Random', alpha=0.7)
ax_spars.set_xlabel('Fraction of pixels removed')
ax_spars.set_ylabel('Residual MSE')
ax_spars.set_title('Sparsification Analysis')
ax_spars.legend(fontsize=7, loc='upper right')
ax_spars.grid(True, alpha=0.3)

ax_scatter = fig.add_subplot(gs[2, 2:])
n_subsample = 5000
idx_sub = np.random.choice(mc_stds_all.flatten().shape[0], n_subsample, replace=False)
ax_scatter.scatter(mc_stds_all.flatten()[idx_sub], errors_mc_all.flatten()[idx_sub],
                   alpha=0.15, s=3, c='blue', label=f'MC Dropout (r={mc_pearson:.3f})')
ax_scatter.scatter(ens_stds_all.flatten()[idx_sub], errors_ens_all.flatten()[idx_sub],
                   alpha=0.15, s=3, c='red', label=f'Ensemble (r={ens_pearson:.3f})')
ax_scatter.set_xlabel('Predicted Uncertainty (std)')
ax_scatter.set_ylabel('Absolute Error')
ax_scatter.set_title('Uncertainty vs. Error Correlation')
ax_scatter.legend(fontsize=7, markerscale=5)
ax_scatter.grid(True, alpha=0.3)

fig.savefig(f'{FIG_DIR}/fig13_ensemble_vs_mcdropout.pdf', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/fig13_ensemble_vs_mcdropout.png', bbox_inches='tight', dpi=300)
plt.close(fig)
print("Saved fig13_ensemble_vs_mcdropout")

Saved fig13_ensemble_vs_mcdropout


In [28]:
# Figure 14: Reliability Diagrams# Reliability diagrams bin predictions by uncertainty quantile and compare
# the mean predicted uncertainty against the mean observed error in each bin.
# Perfect calibration would show these two quantities tracking each other exactly.
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

for ax, unc_data, err_data, method_name, color in [
    (axes[0], mc_stds_all, errors_mc_all, 'MC Dropout', 'blue'),
    (axes[1], ens_stds_all, errors_ens_all, 'Deep Ensemble', 'red')
]:
    n_bins = 15
    unc_flat = unc_data.flatten()
    err_flat = err_data.flatten()

    # Bin by uncertainty quantile for equal-count bins
    percentiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(unc_flat, percentiles)

    bin_means_unc = []
    bin_means_err = []
    bin_counts = []

    for b in range(n_bins):
        mask = (unc_flat >= bin_edges[b]) & (unc_flat < bin_edges[b+1] + 1e-10)
        if mask.sum() > 0:
            bin_means_unc.append(unc_flat[mask].mean())
            bin_means_err.append(err_flat[mask].mean())
            bin_counts.append(mask.sum())

    bin_means_unc = np.array(bin_means_unc)
    bin_means_err = np.array(bin_means_err)

    # Normalize for visual comparison
    max_val = max(bin_means_unc.max(), bin_means_err.max())
    bmu_norm = bin_means_unc / (max_val + 1e-10)
    bme_norm = bin_means_err / (max_val + 1e-10)

    ece_val = mc_ece if 'MC' in method_name else ens_ece

    ax.bar(range(n_bins), bme_norm[:n_bins], alpha=0.4, color=color, label='Observed error')
    ax.plot(range(n_bins), bmu_norm[:n_bins], 'o-', color=color, markersize=4,
            linewidth=1.5, label='Predicted uncertainty')
    ax.plot(range(n_bins), range(n_bins), 'k--', alpha=0.3, linewidth=1)
    diag = np.linspace(0, 1, n_bins)
    ax.plot(range(n_bins), diag, 'k--', alpha=0.3, linewidth=1, label='Perfect calibration')
    ax.set_xlabel('Uncertainty bin')
    ax.set_ylabel('Normalised value')
    ax.set_title(f'{method_name} (ECE={ece_val:.4f})')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig14_reliability_diagrams.pdf', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/fig14_reliability_diagrams.png', bbox_inches='tight', dpi=300)
plt.close(fig)
print("Saved fig14_reliability_diagrams")

Saved fig14_reliability_diagrams


In [29]:
# Figure 15: Uncertainty-Segmentation Correlation# This figure demonstrates the clinical relevance of uncertainty maps by showing
# spatial correspondence between reconstruction uncertainty and segmentation failure.
fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))

s = sample_idx
gt = targets_all[s]
mc_rec = mc_means_all[s]
mc_unc = mc_stds_all[s]
seg_err = seg_error_maps[s]

axes[0].imshow(gt, cmap='gray')
axes[0].set_title('Ground Truth')
axes[0].axis('off')

axes[1].imshow(mc_rec, cmap='gray')
axes[1].set_title('Reconstruction')
axes[1].axis('off')

im2 = axes[2].imshow(mc_unc, cmap='magma')
axes[2].set_title('Uncertainty Map')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

im3 = axes[3].imshow(seg_err, cmap='Reds')
axes[3].set_title('Segmentation Error')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

# Overlay: uncertainty heatmap with segmentation error contours
axes[4].imshow(gt, cmap='gray', alpha=0.5)
axes[4].imshow(mc_unc, cmap='magma', alpha=0.3)
axes[4].contour(seg_err, levels=[0.5], colors='red', linewidths=0.8)
axes[4].set_title(f'Overlay (r={mean_seg_corr_mc:.3f})')
axes[4].axis('off')

fig.suptitle('Reconstruction Uncertainty Predicts Segmentation Failures', fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig15_uncertainty_seg_correlation.pdf', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/fig15_uncertainty_seg_correlation.png', bbox_inches='tight', dpi=300)
plt.close(fig)
print("Saved fig15_uncertainty_seg_correlation")

Saved fig15_uncertainty_seg_correlation


## 7. Results Summary

All quantitative results are saved to a JSON file for downstream use
(e.g., automatic LaTeX table generation via `src/update_latex_results.py`).

In [30]:
# Save structured resultsresults = {
    'mc_dropout': {
        'psnr_mean': float(np.mean(all_psnr_mc)),
        'psnr_std': float(np.std(all_psnr_mc)),
        'ssim_mean': float(np.mean(all_ssim_mc)),
        'ssim_std': float(np.std(all_ssim_mc)),
        'ece': float(mc_ece),
        'ause': float(mc_ause),
        'pearson_r': float(mc_pearson),
        'seg_correlation': float(mean_seg_corr_mc),
    },
    'ensemble': {
        'psnr_mean': float(np.mean(all_psnr_ens)),
        'psnr_std': float(np.std(all_psnr_ens)),
        'ssim_mean': float(np.mean(all_ssim_mc)),
        'ssim_std': float(np.std(all_ssim_ens)),
        'ece': float(ens_ece),
        'ause': float(ens_ause),
        'pearson_r': float(ens_pearson),
        'seg_correlation': float(mean_seg_corr_ens),
    },
    'dc_ablation': {
        'with_dc_psnr': float(np.mean(dc_psnrs)),
        'with_dc_ssim': float(np.mean(dc_ssims)),
        'without_dc_psnr': float(np.mean(nodc_psnrs)),
        'without_dc_ssim': float(np.mean(nodc_ssims)),
        'dc_psnr_gain': float(np.mean(dc_psnrs) - np.mean(nodc_psnrs)),
    }
}

with open(f'{CKPT_DIR}/ensemble_comparison_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n=== FINAL RESULTS ===")
print(json.dumps(results, indent=2))
print("\nAll figures saved to:", FIG_DIR)
print("Results saved to:", f'{CKPT_DIR}/ensemble_comparison_results.json')


=== FINAL RESULTS ===
{
  "mc_dropout": {
    "psnr_mean": 31.946712469650528,
    "psnr_std": 1.4133396272273533,
    "ssim_mean": 0.8893462053294909,
    "ssim_std": 0.015789486164204846,
    "ece": 0.034200264066641936,
    "ause": 9.32110928626036e-05,
    "pearson_r": 0.5901392280156623,
    "seg_correlation": 0.19263961306574656
  },
  "ensemble": {
    "psnr_mean": 32.09440146462392,
    "psnr_std": 1.4531831080034157,
    "ssim_mean": 0.8893462053294909,
    "ssim_std": 0.015904404620385268,
    "ece": 0.01697862341415751,
    "ause": 0.00012922971713430237,
    "pearson_r": 0.4933628865057568,
    "seg_correlation": 0.15641668958837376
  },
  "dc_ablation": {
    "with_dc_psnr": 31.89923383421817,
    "with_dc_ssim": 0.8885842243493614,
    "without_dc_psnr": 23.840043657917086,
    "without_dc_ssim": 0.6191822424278421,
    "dc_psnr_gain": 8.059190176301083
  }
}

All figures saved to: /root/IX-Medical-Imaging/latex/figures
Results saved to: /root/IX-Medical-Imaging/checkpoi